In [ ]:
!git clone --branch Project_Update2 --single-branch \
  https://github.com/GustavoPCS/Detecting-AI-Generated-Text-Across-Domains-and-Generators---NLP-Project.git

%cd Detecting-AI-Generated-Text-Across-Domains-and-Generators---NLP-Project

!git branch --show-current
!pip -q install transformers accelerate pandas numpy scikit-learn datasets pytest

Cloning into 'Detecting-AI-Generated-Text-Across-Domains-and-Generators---NLP-Project'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 74 (delta 19), reused 54 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (74/74), 525.64 KiB | 2.01 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/Detecting-AI-Generated-Text-Across-Domains-and-Generators---NLP-Project
Project_Update1


In [ ]:
# Download the original RAID input and recreate data/processed/dataset.csv
!mkdir -p data/raw
!curl -L -o data/raw/train_none.csv https://dataset.raid-bench.xyz/train_none.csv

!python -m src.build_dataset build \
  --csv data/raw/train_none.csv \
  --decoding greedy \
  --rep-penalty no

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  764M  100  764M    0     0   114M      0  0:00:06  0:00:06 --:--:--  121M
label                         human  machine
domain    generator    split                
abstracts llama-chat   test      46       46
                       train    244      244
                       val       55       55
          mistral-chat test      64       64
                       train    342      342
                       val       77       77
news      llama-chat   test     134      134
                       train    617      617
                       val      133      133
          mistral-chat test     138      138
                       train    646      646
                       val      133      133
reddit    llama-chat   test      85       85
                       train    375      375
                       val       79       7

In [ ]:
!nvidia-smi

from pathlib import Path
import json
import gc
import time

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

DATA_PATH = Path("data/processed/dataset.csv")
OUT_DIR = Path("results/fast_detectgpt")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DOMAINS = ["abstracts", "news", "reviews", "reddit"]
TRAIN_GENERATOR = "mistral-chat"
TEST_GENERATORS = ["mistral-chat", "llama-chat"]

assert DATA_PATH.exists(), f"Missing: {DATA_PATH}"
assert torch.cuda.is_available(), "Enable a GPU runtime in Colab first."

df = pd.read_csv(DATA_PATH)
display(df.groupby(["domain", "generator", "split", "label"]).size())

Sat Nov 21 01:01:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

domain     generator     split  label
abstracts  llama-chat    test   0         46
                                1         46
                         train  0        244
                                1        244
                         val    0         55
                                1         55
           mistral-chat  test   0         64
                                1         64
                         train  0        342
                                1        342
                         val    0         77
                                1         77
news       llama-chat    test   0        134
                                1        134
                         train  0        617
                                1        617
                         val    0        133
                                1        133
           mistral-chat  test   0        138
                                1        138
                         train  0        646
                                1        646
                         val    0        133
                                1        133
reddit     llama-chat    test   0         85
                                1         85
                         train  0        375
                                1        375
                         val    0         79
                                1         79
           mistral-chat  test   0         99
                                1         99
                         train  0        453
                                1        453
                         val    0         93
                                1         93
reviews    llama-chat    test   0         87
                                1         87
                         train  0        463
                                1        463
                         val    0         90
                                1         90
           mistral-chat  test   0        108
                                1        108
                         train  0        423
                                1        423
                         val    0         90
                                1         90
dtype: int64

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

PRIMARY_CHECKPOINT = "EleutherAI/gpt-neo-2.7B"
FALLBACK_CHECKPOINT = "EleutherAI/gpt-neo-1.3B"

def is_cuda_oom(error):
    return "out of memory" in str(error).lower()

attempts = []
model = None
tokenizer = None
selected_checkpoint = None

for checkpoint in [PRIMARY_CHECKPOINT, FALLBACK_CHECKPOINT]:
    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()

    try:
        tokenizer = AutoTokenizer.from_pretrained(checkpoint)

        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token

        model = AutoModelForCausalLM.from_pretrained(
            checkpoint,
            torch_dtype=torch.float16
        ).to("cuda").eval()

        load_seconds = time.perf_counter() - started
        peak_memory_gib = torch.cuda.max_memory_allocated() / 2**30

        attempts.append({
            "checkpoint": checkpoint,
            "status": "loaded",
            "load_seconds": load_seconds,
            "peak_gpu_memory_gib_load": peak_memory_gib
        })

        selected_checkpoint = checkpoint
        break

    except RuntimeError as error:
        attempts.append({
            "checkpoint": checkpoint,
            "status": "failed",
            "error": str(error),
            "load_seconds": time.perf_counter() - started
        })

        torch.cuda.empty_cache()
        gc.collect()

        if not is_cuda_oom(error):
            raise

if model is None:
    raise RuntimeError("Neither Fast-DetectGPT checkpoint could be loaded.")

with open(OUT_DIR / "load_attempts.json", "w") as f:
    json.dump(attempts, f, indent=2)

hardware = {
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_total_memory_gib": round(
        torch.cuda.get_device_properties(0).total_memory / 2**30, 2
    ),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "selected_checkpoint": selected_checkpoint
}

with open(OUT_DIR / "hardware.json", "w") as f:
    json.dump(hardware, f, indent=2)

print(json.dumps(attempts, indent=2))
print("\nUsing:", selected_checkpoint)

config.json:   0%|          | 0.00/1.46k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 10.7GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/420 [00:00<?, ?it/s]

[transformers] GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-2.7B
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
transformer.h.{0...31}.attn.attention.masked_bias | UNEXPECTED |  | 
transformer.h.{0...30}.attn.attention.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[
  {
    "checkpoint": "EleutherAI/gpt-neo-2.7B",
    "status": "loaded",
    "load_seconds": 188.795385463,
    "peak_gpu_memory_gib_load": 5.064035415649414
  }
]

Using: EleutherAI/gpt-neo-2.7B


In [ ]:
def fast_detectgpt_score(text, max_length=256):
    """
    Higher score = more likely machine-generated.
    Uses the analytic discrepancy statistic.
    """
    inputs = tokenizer(
        text,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

    input_ids = inputs.input_ids.to("cuda")

    if input_ids.shape[1] < 2:
        raise ValueError("Text has fewer than two tokens after tokenization.")

    with torch.inference_mode():
        logits = model(
            input_ids=input_ids,
            attention_mask=torch.ones_like(input_ids)
        ).logits[:, :-1].float()

        log_probs = torch.log_softmax(logits, dim=-1)
        observed_tokens = input_ids[:, 1:]

        observed_logp = log_probs.gather(
            -1,
            observed_tokens.unsqueeze(-1)
        ).squeeze(-1)

        probs = log_probs.exp()
        expected_logp = (probs * log_probs).sum(-1)

        variance = (
            probs * (log_probs - expected_logp.unsqueeze(-1)).square()
        ).sum(-1).clamp_min(1e-8)

        score = (
            (observed_logp - expected_logp).sum(-1)
            / variance.sum(-1).sqrt()
        )

    return score.item()


def score_dataframe(rows, label="Scoring"):
    started = time.perf_counter()
    torch.cuda.reset_peak_memory_stats()

    scores = [
        fast_detectgpt_score(text)
        for text in tqdm(rows["text"].tolist(), desc=label)
    ]

    seconds = time.perf_counter() - started
    peak_gib = torch.cuda.max_memory_allocated() / 2**30

    return np.asarray(scores), seconds, peak_gib

In [ ]:
pilot_source = df[
    (df["domain"] == "news") &
    (df["generator"] == "mistral-chat") &
    (df["split"] == "test")
].copy()

n_per_class = min(16, *(len(pilot_source[pilot_source["label"] == y]) for y in [0, 1]))

pilot_rows = pd.concat([
    pilot_source[pilot_source["label"] == 0].sample(n_per_class, random_state=42),
    pilot_source[pilot_source["label"] == 1].sample(n_per_class, random_state=42)
], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

pilot_scores, pilot_seconds, pilot_peak_gib = score_dataframe(
    pilot_rows,
    label="32-text news pilot"
)

pilot_predictions = pilot_rows[
    ["source_id", "label", "domain", "generator", "split"]
].copy()

pilot_predictions.insert(2, "method", "fast_detectgpt_analytic_pilot")
pilot_predictions["score"] = pilot_scores

pilot_out = OUT_DIR / "news_32_pilot"
pilot_out.mkdir(exist_ok=True)

pilot_predictions.to_csv(pilot_out / "predictions.csv", index=False)

from sklearn.metrics import roc_auc_score

pilot_summary = {
    "checkpoint": selected_checkpoint,
    "n_texts": len(pilot_predictions),
    "auroc": roc_auc_score(pilot_predictions["label"], pilot_predictions["score"]),
    "inference_seconds": pilot_seconds,
    "inference_ms_per_1000_texts": 1e6 * pilot_seconds / len(pilot_predictions),
    "peak_gpu_memory_gib_inference": pilot_peak_gib
}

with open(pilot_out / "run_config.json", "w") as f:
    json.dump(pilot_summary, f, indent=2)

print(json.dumps(pilot_summary, indent=2))
display(pilot_predictions.head())

32-text news pilot:   0%|          | 0/32 [00:00<?, ?it/s]

{
  "checkpoint": "EleutherAI/gpt-neo-2.7B",
  "n_texts": 32,
  "auroc": 1.0,
  "inference_seconds": 3.816292934999865,
  "inference_ms_per_1000_texts": 119259.15421874578,
  "peak_gpu_memory_gib_inference": 5.310683250427246
}


,source_id,label,method,domain,generator,split,score
0,b562dfdb-c2b4-4ccf-9259-7eef9990430a,1,fast_detectgpt_analytic_pilot,news,mistral-chat,test,6.868196
1,14cee7ae-fea4-4aef-8495-21bada19789a,0,fast_detectgpt_analytic_pilot,news,mistral-chat,test,0.148895
2,a3aca5c1-f1e0-404e-911c-4207bcce45bc,1,fast_detectgpt_analytic_pilot,news,mistral-chat,test,6.547783
3,4edb58ea-ccd7-4af2-aeac-8ae91754a1d2,1,fast_detectgpt_analytic_pilot,news,mistral-chat,test,6.765961
4,81bc0b1e-2491-46fc-b097-ff2e833ddd45,0,fast_detectgpt_analytic_pilot,news,mistral-chat,test,0.846653


In [ ]:
METHOD = "fast_detectgpt_analytic"

val_frames = []
test_cache = {}
runtime_rows = []

# Score validation data for each calibration domain.
for train_domain in DOMAINS:
    val_rows = df[
        (df["domain"] == train_domain) &
        (df["generator"] == TRAIN_GENERATOR) &
        (df["split"] == "val")
    ].reset_index(drop=True)

    scores, seconds, peak_gib = score_dataframe(
        val_rows,
        label=f"Validation: {train_domain}"
    )

    frame = val_rows[
        ["source_id", "label", "domain", "generator", "split"]
    ].copy()

    frame.insert(2, "method", METHOD)
    frame["train_domain"] = train_domain
    frame["train_generator"] = TRAIN_GENERATOR
    frame["score"] = scores
    val_frames.append(frame)

    runtime_rows.append({
        "method": METHOD,
        "checkpoint": selected_checkpoint,
        "train_domain": train_domain,
        "train_generator": TRAIN_GENERATOR,
        "phase": "validation",
        "n_texts": len(val_rows),
        "seconds": seconds,
        "ms_per_1000_texts": 1e6 * seconds / len(val_rows),
        "peak_gpu_memory_gib": peak_gib
    })

# Score every unique held-out test condition once.
for domain in DOMAINS:
    for generator in TEST_GENERATORS:
        test_rows = df[
            (df["domain"] == domain) &
            (df["generator"] == generator) &
            (df["split"] == "test")
        ].reset_index(drop=True)

        scores, seconds, peak_gib = score_dataframe(
            test_rows,
            label=f"Test: {domain} / {generator}"
        )

        test_cache[(domain, generator)] = (test_rows, scores)

        runtime_rows.append({
            "method": METHOD,
            "checkpoint": selected_checkpoint,
            "train_domain": None,
            "train_generator": TRAIN_GENERATOR,
            "phase": f"test_{domain}_{generator}",
            "n_texts": len(test_rows),
            "seconds": seconds,
            "ms_per_1000_texts": 1e6 * seconds / len(test_rows),
            "peak_gpu_memory_gib": peak_gib
        })

print("Finished scoring validation and test data.")

Validation: abstracts:   0%|          | 0/154 [00:00<?, ?it/s]

Validation: news:   0%|          | 0/266 [00:00<?, ?it/s]

Validation: reviews:   0%|          | 0/180 [00:00<?, ?it/s]

Validation: reddit:   0%|          | 0/186 [00:00<?, ?it/s]

Test: abstracts / mistral-chat:   0%|          | 0/128 [00:00<?, ?it/s]

Test: abstracts / llama-chat:   0%|          | 0/92 [00:00<?, ?it/s]

... (3 outputs left out)

Test: reviews / llama-chat:   0%|          | 0/174 [00:00<?, ?it/s]

Test: reddit / mistral-chat:   0%|          | 0/198 [00:00<?, ?it/s]

Test: reddit / llama-chat:   0%|          | 0/170 [00:00<?, ?it/s]

Finished scoring validation and test data.


In [ ]:
from src import evaluate as E

# The same raw test scores are evaluated under each domain's
# validation-selected, frozen threshold.
test_frames = []

for train_domain in DOMAINS:
    for (domain, generator), (test_rows, scores) in test_cache.items():
        frame = test_rows[
            ["source_id", "label", "domain", "generator", "split"]
        ].copy()

        frame.insert(2, "method", METHOD)
        frame["train_domain"] = train_domain
        frame["train_generator"] = TRAIN_GENERATOR
        frame["score"] = scores
        test_frames.append(frame)

predictions_val = pd.concat(val_frames, ignore_index=True)
predictions_test = pd.concat(test_frames, ignore_index=True)
runtime = pd.DataFrame(runtime_rows)

predictions_val.to_csv(OUT_DIR / "predictions_val.csv", index=False)
predictions_test.to_csv(OUT_DIR / "predictions_test.csv", index=False)
runtime.to_csv(OUT_DIR / "runtime.csv", index=False)

# Threshold is selected only on each G1 validation domain, then frozen for test.
metrics = E.run(
    predictions_test,
    predictions_val,
    n_boot=1000,
    seed=42
)

metrics.to_csv(OUT_DIR / "metrics.csv", index=False)

display(metrics[[
    "method",
    "train_domain",
    "domain",
    "generator",
    "n",
    "threshold",
    "macro_f1",
    "auroc",
    "fpr",
    "gap_macro_f1"
]].round(3))

display(runtime.round(3))

In [ ]:
from google.colab import files
import shutil

archive_name = "fast_detectgpt_results"

shutil.make_archive(
    archive_name,
    "zip",
    root_dir=OUT_DIR
)

files.download(f"{archive_name}.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>